In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,col,expr,when
from pyspark.sql.types import ArrayType, IntegerType, ShortType

In [2]:
spark = SparkSession.builder.appName('ETL').config("spark.driver.memory", "4g").getOrCreate()
spark

25/04/30 13:53:23 WARN Utils: Your hostname, namunaacharya resolves to a loopback address: 127.0.1.1; using 192.168.1.190 instead (on interface wlx60fb0067d846)
25/04/30 13:53:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/30 13:53:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
rate_file = spark.read.option('multiline','True').json('files/rate.json')
provider_file = spark.read.option('multiline','True').json('files/provider.json')

Flatten provider file

In [4]:
provider_file.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- provider_groups: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- npi: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- tin: struct (nullable = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |    |-- value: string (nullable = true)



In [5]:
provider_df = provider_file.withColumn("provider", explode("provider_groups"))
provider_npi = provider_df.withColumn("provider_npi", explode("provider.npi"))

provider_flat = provider_npi.select(
    "provider_group_id",
    col("provider_npi").alias("npi"),
    col("provider.tin.type").alias("tin_type"),
    col("provider.tin.value").alias("tin")
)

provider_flat.show(truncate=False)

+-----------------+----------+--------+----------+
|provider_group_id|npi       |tin_type|tin       |
+-----------------+----------+--------+----------+
|10001001         |1235233008|ein     |04-3267217|
|10001001         |1316041189|ein     |04-3267217|
|10001001         |1780788554|ein     |04-3267217|
|10001001         |1891068409|ein     |04-3267217|
|10001001         |1366459570|ein     |11-1562701|
|10001001         |1417915653|ein     |11-3358535|
|10001001         |1417915653|ein     |13-3888838|
|10002001         |1609829761|ein     |00-0004110|
|10002001         |1821482241|ein     |00-0004110|
|10002001         |1760986277|ein     |00-6980743|
|10002001         |1215075882|ein     |01-0550744|
|10002001         |1013917665|ein     |01-0555304|
|10002001         |1679780811|ein     |01-0555483|
|10002001         |1700093952|ein     |01-0555483|
|10002001         |1780072447|ein     |01-0555483|
|10002001         |1952532970|ein     |01-0555483|
|10002001         |1376647511|e

In [6]:
provider_flat.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)



Flatten rate file

In [7]:
rate_file.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiated_rates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- additional_information: string (nullable = true)
 |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |-- billing_code_modifier: array (nullable = true)
 |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |-- expiration_date: string (nullable = true)
 |    |    |    |    |-- negotiated_rate: double (nullable = true)
 |    |    |    |    |-- negotiated_type: string (nullable = true)
 |    |    |    |    |-- service_code: array (nullable = true)
 |    |    |

In [8]:
rate_df = rate_file.withColumn("rates", explode("negotiated_rates"))
id_df = rate_df.withColumn("provider_group_id",explode("rates.provider_references"))
price_df = id_df.withColumn("prices",explode("rates.negotiated_prices"))

rate_flat = price_df.select(
    "billing_code",
    "billing_code_type",
    "negotiation_arrangement",
    col("provider_group_id").alias("provider_group_id"),    
    col("prices.billing_class").alias("billing_class"),
    col("prices.billing_code_modifier").alias("billing_code_modifier"),
    col("prices.negotiated_rate").alias("negotiated_rate"),
    col("prices.negotiated_type").alias("negotiated_type"),
    col("prices.service_code").alias("service_code")
)
rate_flat.show()

+------------+-----------------+-----------------------+-----------------+-------------+---------------------+---------------+---------------+------------+
|billing_code|billing_code_type|negotiation_arrangement|provider_group_id|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|
+------------+-----------------+-----------------------+-----------------+-------------+---------------------+---------------+---------------+------------+
|       PEMG1|              CPT|                    ffs|         10001001| professional|                 NULL|           70.0|     negotiated|        [11]|
|       PEMG1|              CPT|                    ffs|         10002001| professional|                 NULL|           70.0|     negotiated|        [11]|
|       PEMG1|              CPT|                    ffs|         10003001| professional|                 NULL|           70.0|     negotiated|        [11]|
|       PEMG1|              CPT|                    ffs|        

In [9]:
rate_flat.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- provider_group_id: long (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)



Cleaning rate file

In [10]:
rate_filtered=rate_flat.filter(rate_flat.billing_code.isNotNull())

rate_filtered.show()

+------------+-----------------+-----------------------+-----------------+-------------+---------------------+---------------+---------------+------------+
|billing_code|billing_code_type|negotiation_arrangement|provider_group_id|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|
+------------+-----------------+-----------------------+-----------------+-------------+---------------------+---------------+---------------+------------+
|       PEMG1|              CPT|                    ffs|         10001001| professional|                 NULL|           70.0|     negotiated|        [11]|
|       PEMG1|              CPT|                    ffs|         10002001| professional|                 NULL|           70.0|     negotiated|        [11]|
|       PEMG1|              CPT|                    ffs|         10003001| professional|                 NULL|           70.0|     negotiated|        [11]|
|       PEMG1|              CPT|                    ffs|        

 Verify null

In [11]:
rate_filtered.filter(rate_filtered.billing_code_modifier.isNotNull()).count()

454512

Cleaning provider file

In [12]:
provider_flat.show()

+-----------------+----------+--------+----------+
|provider_group_id|       npi|tin_type|       tin|
+-----------------+----------+--------+----------+
|         10001001|1235233008|     ein|04-3267217|
|         10001001|1316041189|     ein|04-3267217|
|         10001001|1780788554|     ein|04-3267217|
|         10001001|1891068409|     ein|04-3267217|
|         10001001|1366459570|     ein|11-1562701|
|         10001001|1417915653|     ein|11-3358535|
|         10001001|1417915653|     ein|13-3888838|
|         10002001|1609829761|     ein|00-0004110|
|         10002001|1821482241|     ein|00-0004110|
|         10002001|1760986277|     ein|00-6980743|
|         10002001|1215075882|     ein|01-0550744|
|         10002001|1013917665|     ein|01-0555304|
|         10002001|1679780811|     ein|01-0555483|
|         10002001|1700093952|     ein|01-0555483|
|         10002001|1780072447|     ein|01-0555483|
|         10002001|1952532970|     ein|01-0555483|
|         10002001|1376647511| 

In [13]:
provider_replaced = provider_flat.withColumn("tin_type",
                    when(col("tin_type") == "ein", 1)
                    .when(col("tin_type") == "npi", 2))
provider_cleaned = provider_replaced.withColumn('tin', expr("REPLACE(tin, '-', '')"))
provider_cleaned.show()

+-----------------+----------+--------+---------+
|provider_group_id|       npi|tin_type|      tin|
+-----------------+----------+--------+---------+
|         10001001|1235233008|       1|043267217|
|         10001001|1316041189|       1|043267217|
|         10001001|1780788554|       1|043267217|
|         10001001|1891068409|       1|043267217|
|         10001001|1366459570|       1|111562701|
|         10001001|1417915653|       1|113358535|
|         10001001|1417915653|       1|133888838|
|         10002001|1609829761|       1|000004110|
|         10002001|1821482241|       1|000004110|
|         10002001|1760986277|       1|006980743|
|         10002001|1215075882|       1|010550744|
|         10002001|1013917665|       1|010555304|
|         10002001|1679780811|       1|010555483|
|         10002001|1700093952|       1|010555483|
|         10002001|1780072447|       1|010555483|
|         10002001|1952532970|       1|010555483|
|         10002001|1376647511|       1|010567880|


In [14]:
provider_cleaned.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: integer (nullable = true)
 |-- tin: string (nullable = true)



Cast the datatype of tin type to small integer

In [15]:
provider_cast = provider_cleaned.withColumn("tin_type", col("tin_type").cast(ShortType()))
provider_cast.printSchema()

root
 |-- provider_group_id: long (nullable = true)
 |-- npi: long (nullable = true)
 |-- tin_type: short (nullable = true)
 |-- tin: string (nullable = true)



In [16]:
rate_filtered.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- provider_group_id: long (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)



Cast the datatype of service code to array of integer

In [17]:
rate_cast = rate_filtered.withColumn("service_code",col("service_code").cast(ArrayType(IntegerType())))
rate_cast.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- provider_group_id: long (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [18]:
rate_cast.write.parquet("files/rate_data.parquet")
provider_cast.write.parquet("files/provider_data.parquet")